# Week 9 build · A contrastive model from scratch, then CLIP with error bars

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mjtri/learnml/blob/main/notebooks/week-09.ipynb)

**~3 hours · CPU for Parts A–D, T4 for Parts E–F (Runtime → Change runtime type → T4) · run top to bottom.**

**The one rule:** every experiment is preceded by a 🔮 **predict cell**. Type what you expect and how sure you are *before* running the experiment. The notebook refuses empty predictions. A rough guess is fine; the gap between guess and result is what you are here to collect (canvas steps 7–8: *predict before running*, *log the surprise*).

**Two passes.** First run with `SMOKE = True` (next cell but one): Parts A–D on CPU in about five minutes, Parts E–F skipped. Then set `SMOKE = False`, switch to the T4, and run again: Part E downloads CLIP weights (~600 MB) and CIFAR-10 (~170 MB).

| Part | Topic | Time |
|---|---|---|
| A | 8×8 shapes and their vOICe-style "sound" vectors: pairs for free | 20 min |
| B | Two encoders and InfoNCE from scratch; similarity matrix before and after; held-out retrieval | 40 min |
| C | Ablations: temperature, batch size, and the shuffled-pairing run | 25 min |
| D | Three seeds → mean ± standard error; a bootstrap interval | 25 min |
| E | OpenCLIP zero-shot CIFAR-10 with prompt templates and intervals (T4) | 40 min |
| F | Image↔text retrieval on your own images (T4) | 30 min |

In [ ]:
#@title Setup: run me first
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, Markdown, display
torch.manual_seed(0); np.random.seed(0)
_LEDGER = {}

def predict(tag, prediction, confidence):
    """Log a prediction BEFORE the experiment. Refuses empty predictions."""
    if not str(prediction).strip():
        raise ValueError(f"[{tag}] Write a prediction first. A rough guess is fine; an empty one is not.")
    _LEDGER[tag] = {"prediction": str(prediction).strip(), "confidence": int(confidence), "actual": "", "surprise": ""}
    print(f"Logged [{tag}] at {confidence}% confidence: {prediction}")

def record(tag, what_happened, surprise="none"):
    if tag not in _LEDGER:
        raise ValueError(f"[{tag}] has no prediction yet. Run its predict cell first.")
    _LEDGER[tag].update(actual=str(what_happened).strip(), surprise=surprise)
    print(f"Recorded [{tag}] surprise={surprise}")

def reveal():
    rows = ["| # | prediction | sure | what happened | surprise |", "|---|---|---|---|---|"]
    rows += [f"| {t} | {r['prediction']} | {r['confidence']}% | {r['actual']} | {r['surprise']} |" for t, r in _LEDGER.items()]
    print("\n".join(rows)); display(Markdown("\n".join(rows)))

print("torch", torch.__version__, "| ready")

---
## Part A · Pairs for free (20 min)

**Words used in this part.** *pair*: an image and the sound made from it; the pairing costs nothing because one is computed from the other. *vOICe-style scan*: read the image column by column, left to right; each column becomes a moment of sound with a loudness (how many pixels are lit) and a pitch (how high the lit pixels sit), plus microphone-style noise so the two sides are not trivially identical. *held-out*: pairs made after the training pairs and never trained on.

In [ ]:
#@title Config: SMOKE = True is a five-minute CPU rehearsal of Parts A–D; False adds the T4 parts
SMOKE = True  #@param {type:"boolean"}
import math, time, json, os, sys, torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = dict(n_pairs=512 if SMOKE else 4096, n_test=128 if SMOKE else 512, d=32, batch=32,
           steps=300 if SMOKE else 1500, lr=3e-3, tau=0.1, seed=0)
torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])
print("device:", device, "| config:", cfg)

In [ ]:
def make_shape(kind, rng):
    """One 8x8 image: a square, a cross, a ring or a diagonal at a random place and size."""
    img = np.zeros((8, 8), dtype=np.float32)
    s = int(rng.integers(3, 6)); r, c = (int(v) for v in rng.integers(0, 9 - s, size=2))
    if kind == 0:   img[r:r+s, c:c+s] = 1
    elif kind == 1: img[r + s//2, c:c+s] = 1; img[r:r+s, c + s//2] = 1
    elif kind == 2: img[r:r+s, c:c+s] = 1; img[r+1:r+s-1, c+1:c+s-1] = 0
    else:
        for k in range(s): img[r+k, c+k] = 1
    return img

def make_sound(img, rng, noise=0.15):
    """vOICe-style: per column, loudness (lit fraction) and pitch (mean lit row, 0 = top). 16 numbers."""
    lit = img.sum(0)
    loud = lit / 8.0
    pitch = (np.arange(8)[:, None] * img).sum(0) / np.maximum(lit, 1) / 7.0
    return np.concatenate([loud, pitch]) + noise * rng.standard_normal(16)

def make_pairs(n, rng):
    kinds = rng.integers(0, 4, size=n)
    imgs = np.stack([make_shape(int(k), rng) for k in kinds])
    snds = np.stack([make_sound(im, rng) for im in imgs]).astype(np.float32)
    return torch.tensor(imgs), torch.tensor(snds), torch.tensor(kinds)

In [ ]:
rng = np.random.default_rng(cfg["seed"])
X_img, X_snd, Y = make_pairs(cfg["n_pairs"], rng)        # training pairs
T_img, T_snd, T_y = make_pairs(cfg["n_test"], rng)       # held-out pairs, made afterwards
print("train images", tuple(X_img.shape), "sounds", tuple(X_snd.shape), "| held-out", tuple(T_img.shape))
NAMES = ["square", "cross", "ring", "diagonal"]
fig, ax = plt.subplots(2, 4, figsize=(9, 4))
for j in range(4):
    ax[0, j].imshow(X_img[j], cmap="gray_r"); ax[0, j].set_title(NAMES[int(Y[j])]); ax[0, j].axis("off")
    ax[1, j].bar(range(16), X_snd[j], color=["C0"] * 8 + ["C1"] * 8); ax[1, j].set_ylim(0, 1.1)
    ax[1, j].set_xticks([3.5, 11.5]); ax[1, j].set_xticklabels(["loudness", "pitch"])
plt.tight_layout(); plt.show()
print("Top: the image. Bottom: its sound vector, 8 loudness numbers then 8 pitch numbers, plus noise.")

The sound is *computed from* the image, so every pair is a positive pair for free, and every other item in a batch is a negative. Nothing here is a label: the model never learns the word "square".

---
## Part B · Two encoders and InfoNCE from scratch (40 min)

**Words used in this part.** *encoder*: a small network that turns one input (an image, or a sound vector) into a unit-length vector of `d` numbers. *similarity matrix*: for a batch of B pairs, the B × B table of cosine similarities, images as rows, sounds as columns. *InfoNCE*: cross-entropy on each row of that matrix (divided by the temperature) with "column i" as the answer for row i, plus the same for columns. *retrieval accuracy*: how often the true sound is the most similar one in the batch; chance is 1/B.

In [ ]:
#@title 🔮 Predict B1: an untrained model, batch of 32 held-out pairs. Loss (chance is ln 32 = 3.47) and retrieval accuracy (chance 1/32 = 0.03)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B1", prediction, confidence)

In [ ]:
class Enc(nn.Module):
    def __init__(self, n_in, d):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(n_in, 64), nn.ReLU(), nn.Linear(64, d))
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)          # unit length, so dot product = cosine

def info_nce(a, b, tau):
    logits = a @ b.T / tau                                # (B, B) similarity matrix over temperature
    labels = torch.arange(len(a), device=a.device)        # row i belongs to column i
    return (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels)) / 2

def make_model(seed):
    torch.manual_seed(seed)
    return Enc(64, cfg["d"]).to(device), Enc(16, cfg["d"]).to(device)

def hits(img_enc, snd_enc, imgs=None, snds=None, batch=32):
    """1 if each held-out image picks its own sound out of its batch, else 0."""
    imgs, snds = (T_img if imgs is None else imgs), (T_snd if snds is None else snds); out = []
    with torch.no_grad():
        for i in range(0, len(imgs) - batch + 1, batch):
            a, b = img_enc(imgs[i:i+batch].to(device)), snd_enc(snds[i:i+batch].to(device))
            out.append((a @ b.T).argmax(1).eq(torch.arange(batch, device=device)).cpu().numpy())
    return np.concatenate(out)

In [ ]:
img_enc, snd_enc = make_model(cfg["seed"])
with torch.no_grad():
    a, b = img_enc(T_img[:32].to(device)), snd_enc(T_snd[:32].to(device))
S0 = (a @ b.T).cpu()
print(f"untrained: loss {info_nce(a, b, cfg['tau']).item():.3f}   chance ln 32 = {math.log(32):.3f}")
print(f"untrained: held-out retrieval accuracy {hits(img_enc, snd_enc).mean():.3f}   chance {1/32:.3f}")

In [ ]:
#@title ✍️ B1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict B2: after training, held-out retrieval accuracy in batches of 32? Will the training loss reach 0?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("B2", prediction, confidence)

In [ ]:
def train(seed, tau=None, batch=None, steps=None, shuffle_pairs=False, log_every=100, quiet=False):
    tau, batch, steps = tau or cfg["tau"], batch or cfg["batch"], steps or cfg["steps"]
    img_enc, snd_enc = make_model(seed)
    opt = torch.optim.Adam(list(img_enc.parameters()) + list(snd_enc.parameters()), lr=cfg["lr"])
    g = torch.Generator().manual_seed(seed)
    snd = X_snd[torch.randperm(len(X_snd), generator=g)] if shuffle_pairs else X_snd   # break the pairing
    for step in range(1, steps + 1):
        idx = torch.randint(0, len(X_img), (batch,), generator=g)
        loss = info_nce(img_enc(X_img[idx].to(device)), snd_enc(snd[idx].to(device)), tau)
        opt.zero_grad(); loss.backward(); opt.step()
        if not quiet and step % log_every == 0:
            print(f"step {step:5d}  loss {loss.item():.3f}")
    return img_enc, snd_enc

t0 = time.time()
img_enc, snd_enc = train(cfg["seed"])
h = hits(img_enc, snd_enc)
print(f"held-out retrieval accuracy {h.mean():.3f} ({int(h.sum())} of {len(h)})   chance {1/32:.3f}   [{time.time()-t0:.0f} s]")

In [ ]:
with torch.no_grad():
    S1 = (img_enc(T_img[:32].to(device)) @ snd_enc(T_snd[:32].to(device)).T).cpu()
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a_, S, t in [(ax[0], S0, "untrained"), (ax[1], S1, "trained")]:
    a_.imshow(S[:16, :16], cmap="viridis", vmin=-1, vmax=1); a_.set_title(f"{t}: 16 held-out pairs")
    a_.set_xlabel("sound"); a_.set_ylabel("image")
plt.tight_layout(); plt.show()
print("A bright diagonal is the goal: image i most similar to sound i. Off-diagonal bright cells are hard negatives:")
for i in range(16):
    j = int(S1[i, :16].argmax())
    if j != i: print(f"  image {i} ({NAMES[int(T_y[i])]}) picked sound {j} ({NAMES[int(T_y[j])]})")

In [ ]:
#@title ✍️ B2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("B2", what_happened, surprise)

Surprise? Same-shape pairs at nearly the same place are the hard negatives: the sound vectors are almost identical, so the batch contains near-duplicates and the loss cannot reach zero. That is a property of the *data*, not a bug.

---
## Part C · Ablations: temperature, batch size, shuffled pairing (25 min)

**Words used in this part.** *temperature τ*: the number the similarities are divided by before softmax; small τ makes the loss care mostly about the hardest negatives. *batch size*: how many pairs are compared at once, so B − 1 negatives per item. *shuffled pairing*: train with every image matched to a random sound; there is nothing to learn, so the held-out score must fall to chance. The dumbest comparison, run on purpose.

In [ ]:
#@title 🔮 Predict C1: temperatures 0.02, 0.1, 0.5, 2.0 for the same steps: which trains best, which worst? Does the shuffled-pairing run land at chance (0.03)?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C1", prediction, confidence)

In [ ]:
rows = []
for tau in [0.02, 0.1, 0.5, 2.0]:
    acc = hits(*train(cfg["seed"], tau=tau, quiet=True)).mean()
    rows.append((f"tau {tau}", acc))
rows.append(("shuffled pairing, tau 0.1", hits(*train(cfg["seed"], shuffle_pairs=True, quiet=True)).mean()))
print(f"{'run':28s} held-out retrieval accuracy   (chance {1/32:.3f})")
for name, acc in rows:
    print(f"{name:28s} {acc:.3f}  {'#' * int(acc * 40)}")

In [ ]:
#@title ✍️ C1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict C2: batch sizes 8, 32, 128 for the same number of steps, all scored in held-out batches of 32: which wins, and by how much?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("C2", prediction, confidence)

In [ ]:
print(f"{'batch':10s} negatives/item   held-out accuracy (scored in batches of 32)")
for B in [8, 32, 128]:
    acc = hits(*train(cfg["seed"], batch=B, quiet=True)).mean()
    print(f"{B:<10d} {B-1:<15d} {acc:.3f}  {'#' * int(acc * 40)}")
print("Same steps means the batch-128 run also saw 4x more examples: a fair comparison would fix compute, not steps.")

In [ ]:
#@title ✍️ C2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("C2", what_happened, surprise)

Surprise? Too hot (2.0) and every negative counts the same, so the loss barely distinguishes hard from easy and the model learns slowly. The coldest setting still works *here* because this toy has few near-duplicate negatives; with 400 million web captions there are many, which is why CLIP learns its temperature and clamps it above 0.01. The shuffled run is the check that the number means anything: if it did *not* fall to chance, the model would be reading something other than the pairing. And more negatives per step (batch 128) is not automatically better at a fixed step count: look at the numbers, not the story.

---
## Part D · Uncertainty: seeds and the bootstrap (25 min)

**Words used in this part.** *seed*: the number that fixes the random initialization and the batch order; one seed is one participant. *standard error*: the spread of the seeds' scores divided by the square root of how many seeds. *bootstrap*: resample the held-out hits with replacement, recompute accuracy, a thousand times; the middle 95% of those is the interval. *confidence interval*: a range, not a number; report it.

In [ ]:
#@title 🔮 Predict D1: three seeds of the same config: how far apart will the held-out accuracies be (in points)? Will tau 0.1 beat tau 0.5 by more than twice the standard error of the difference?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D1", prediction, confidence)

In [ ]:
def seeds_summary(name, tau, seeds=(0, 1, 2)):
    accs = np.array([hits(*train(s, tau=tau, quiet=True)).mean() for s in seeds])
    se = accs.std(ddof=1) / np.sqrt(len(accs))
    print(f"{name:8s} seeds {accs.round(3)} -> {accs.mean():.3f} ± {se:.3f} (standard error, {len(accs)} seeds)")
    return accs.mean(), se

mA, seA = seeds_summary("tau 0.1", 0.1)
mB, seB = seeds_summary("tau 0.5", 0.5)
se_diff = math.hypot(seA, seB)
print(f"difference {mA - mB:+.3f} ± {se_diff:.3f}  ->  {abs(mA - mB) / se_diff:.1f} standard errors apart")
print("Rule of thumb: claim a difference only when it is more than about two standard errors.")

In [ ]:
#@title ✍️ D1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict D2: bootstrap 95% interval on the seed-0 model's held-out hits (n = number of held-out pairs): how wide, in points? Guess with sqrt(p(1-p)/n) first.
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("D2", prediction, confidence)

In [ ]:
def boot_ci(h, n_boot=1000, seed=0):
    """The lesson's ten lines: resample with replacement, recompute, take the middle 95%."""
    rng = np.random.default_rng(seed)
    boot = [rng.choice(h, len(h), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(boot, [2.5, 97.5]), np.array(boot)

h = hits(img_enc, snd_enc)                      # the Part B model, seed 0
(lo, hi), boot = boot_ci(h)
p, n = h.mean(), len(h)
se = math.sqrt(p * (1 - p) / n)
print(f"n = {n}, accuracy {p:.3f}")
print(f"bootstrap 95% interval  [{lo:.3f}, {hi:.3f}]  width {hi - lo:.3f}")
print(f"formula p ± 1.96·sqrt(p(1-p)/n)  [{p - 1.96*se:.3f}, {p + 1.96*se:.3f}]  width {2*1.96*se:.3f}")
plt.hist(boot, bins=30, color="C0"); plt.axvline(lo, color="k"); plt.axvline(hi, color="k"); plt.axvline(p, color="C1")
plt.title("1000 bootstrap accuracies; black = 95% interval, orange = observed"); plt.show()

In [ ]:
#@title ✍️ D2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("D2", what_happened, surprise)

Surprise? Two kinds of uncertainty, two tools. Seeds answer "would a re-trained model score the same?"; the bootstrap answers "would a different held-out set score the same?". A results table needs the one that matches the claim, and often both.

---
## Part E · Zero-shot CIFAR-10 with OpenCLIP (40 min · T4)

**Words used in this part.** *OpenCLIP*: an open reimplementation of CLIP with downloadable weights; `ViT-B-32` is the smallest common one. *zero-shot*: no CIFAR-10 training; the ten class names, written as sentences, are the classifier. *prompt template*: the sentence each class name is dropped into. *CIFAR-10*: 10,000 tiny (32×32) test images in ten classes; we score 2,000 of them.

**GPU needed.** This part is skipped when `SMOKE = True` or no GPU is present. On the T4 it downloads ~600 MB of weights and ~170 MB of images, then runs in a few minutes.

In [ ]:
RUN_CLIP = (not SMOKE) and torch.cuda.is_available()
if not SMOKE and not torch.cuda.is_available():
    print("SMOKE = False but no GPU: Parts E-F would take about an hour on CPU. Runtime -> Change runtime type -> T4.")
if RUN_CLIP:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch", "torchvision"], check=True)
    import open_clip, torchvision
    print("downloading ViT-B-32 weights (~600 MB) and CIFAR-10 (~170 MB) ...")
    model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model = model.to(device).eval()
    cifar = torchvision.datasets.CIFAR10(root="data", train=False, download=True, transform=preprocess)
    cifar_raw = torchvision.datasets.CIFAR10(root="data", train=False, download=False)
    N_EVAL = 2000
    print("ready:", len(cifar), "test images; scoring", N_EVAL, "| classes:", cifar.classes)
else:
    print("Part E skipped (SMOKE = True, or no GPU).")

In [ ]:
#@title 🔮 Predict E1: zero-shot CIFAR-10 accuracy with bare class names, and with 'a photo of a {}.'? Which of four templates wins, and do their intervals overlap?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E1", prediction, confidence)

In [ ]:
if RUN_CLIP:
    TEMPLATES = {"bare": "{}", "photo": "a photo of a {}.", "blurry": "a blurry photo of a {}.",
                 "lowres": "a low resolution photo of a {}."}
    loader = torch.utils.data.DataLoader(torch.utils.data.Subset(cifar, range(N_EVAL)), batch_size=250)
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            feats.append(F.normalize(model.encode_image(x.to(device)), dim=-1)); labels.append(y)
    feats, labels = torch.cat(feats), torch.cat(labels).to(device)
    def text_vectors(template):
        with torch.no_grad():
            return F.normalize(model.encode_text(tokenizer([template.format(c) for c in cifar.classes]).to(device)), dim=-1)
    def zero_shot_hits(t):
        return (feats @ t.T).argmax(1).eq(labels).float().cpu().numpy()
    hits_by = {k: zero_shot_hits(text_vectors(v)) for k, v in TEMPLATES.items()}
    print(f"{'template':10s} accuracy  95% bootstrap interval   (n = {N_EVAL})")
    for k, hk in hits_by.items():
        (lo, hi), _ = boot_ci(hk)
        print(f"{k:10s} {hk.mean():.3f}     [{lo:.3f}, {hi:.3f}]")

In [ ]:
#@title ✍️ E1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E1", what_happened, surprise)

In [ ]:
#@title 🔮 Predict E2: average the four templates' text vectors into one classifier: better than the best single template, or inside its interval? Guess the paired difference in points.
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("E2", prediction, confidence)

In [ ]:
if RUN_CLIP:
    ens = F.normalize(sum(text_vectors(v) for v in TEMPLATES.values()), dim=-1)     # one vector per class
    h_ens = zero_shot_hits(ens)
    best = max(hits_by, key=lambda k: hits_by[k].mean())
    diff = h_ens - hits_by[best]                                   # paired: same images, per-image difference
    (lo, hi), _ = boot_ci(diff)
    print(f"ensemble {h_ens.mean():.3f}   best single ({best}) {hits_by[best].mean():.3f}")
    print(f"paired difference {diff.mean():+.4f}, 95% interval [{lo:+.4f}, {hi:+.4f}]")
    print("If the interval contains 0, the ensemble has not been shown to help on these 2,000 images.")

In [ ]:
#@title ✍️ E2: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("E2", what_happened, surprise)

Surprise? The paired bootstrap resamples *images*, and the difference is computed per image, so it is far tighter than comparing two separate intervals. That is the same trick as a within-participant design.

---
## Part F · Image↔text retrieval on your own images (30 min · T4)

**Words used in this part.** *retrieval*: embed every image once, embed a sentence, sort the images by cosine similarity. *query*: the sentence. *top-k*: the k best matches, which you then check by eye, because a sorted list has no threshold.

Upload your own screenshots (Files panel → new folder `my_images` → drag files in). With no folder, 24 CIFAR-10 images stand in.

In [ ]:
if RUN_CLIP:
    import glob
    from PIL import Image
    files = sorted(glob.glob("my_images/*"))
    if files:
        ims = [Image.open(f).convert("RGB") for f in files]; names = [os.path.basename(f) for f in files]
    else:
        ims = [cifar_raw[i][0] for i in range(24)]; names = [f"cifar {i}: {cifar.classes[cifar_raw[i][1]]}" for i in range(24)]
    with torch.no_grad():
        gallery = F.normalize(model.encode_image(torch.stack([preprocess(im) for im in ims]).to(device)), dim=-1)
    print(len(ims), "images embedded:", names[:6], "...")

In [ ]:
#@title 🔮 Predict F1: for the query 'a photo of a truck' (or your own first query), will all top-3 images be right? Which query will fail?
prediction = ""  #@param {type:"string"}
confidence = 50  #@param {type:"slider", min:0, max:100, step:10}
predict("F1", prediction, confidence)

In [ ]:
QUERIES = "a photo of a truck; a red car; an animal standing in grass"  #@param {type:"string"}
if RUN_CLIP:
    qs = [q.strip() for q in QUERIES.split(";") if q.strip()]
    with torch.no_grad():
        qv = F.normalize(model.encode_text(tokenizer(qs).to(device)), dim=-1)
    sims = (qv @ gallery.T).cpu()
    fig, ax = plt.subplots(len(qs), 3, figsize=(7, 2.4 * len(qs)))
    ax = np.array(ax).reshape(len(qs), 3)
    for qi, q in enumerate(qs):
        top = sims[qi].argsort(descending=True)[:3]
        print(f"{q!r}: " + ", ".join(f"{names[int(j)]} ({sims[qi, j]:.2f})" for j in top))
        for k, j in enumerate(top):
            ax[qi, k].imshow(ims[int(j)]); ax[qi, k].set_title(f"{q[:18]} #{k+1} {sims[qi, j]:.2f}", fontsize=8); ax[qi, k].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("Part F skipped (SMOKE = True, or no GPU).")

In [ ]:
#@title ✍️ F1: what actually happened, and was it a surprise?
what_happened = ""  #@param {type:"string"}
surprise = "none"  #@param ["none", "small", "big"]
record("F1", what_happened, surprise)

Surprise? A cosine of 0.30 can be the best match and still be wrong. The score ranks; it does not decide. For a paper you would count hits in the top k by hand, on images the prompts were not tuned on.

---
## Wrap up

Run the cell below, copy the table into your insight ledger next to `bridge-crossmodal`, and mark the biggest surprise. Then on the laptop:

```
python track.py done week-09/build --rating <1-5> --minutes 180 --note "biggest surprise"
python gen_week.py 10
```

In [ ]:
reveal()